# Chapter 31 — The Interface Is Not the Agent

**Companion to *Pi Agents*** · [`Pi Agents` chapter 31](https://programmer.ie/books/pi/31-chapter/)

| | |
|---|---|
| Chapter | `31-chapter.md` · weight 31 · `/books/pi/31-chapter/` |
| Notebook | `notebooks/pi/31-chapter.ipynb` |
| Pin | `@earendil-works/pi-*` **1.0.4** (`examples/package.json`) |
| Ground truth | `docs/*.md` and `dist/*.d.ts` **at the pin**, under `examples/node_modules/` |

> **Where the authority lives.** Every claim below is checked against the
> packages installed under `examples/node_modules/@earendil-works/`, not against
> a globally installed Pi and not from memory. When prose and an exported
> declaration disagree, the declaration wins.

## The question

Do print, JSON and RPC give the same answer and the same events, with only the
failure contract differing?

The chapter's point: the interface is an **observer**. The same agent runs under
each, produces the same answer and the same lifecycle events. What differs is
what you can watch — print has no events, JSON has no responses, RPC has both —
and how a failure is reported.


## What this notebook establishes

- The same scripted run through **print, JSON and RPC** gives the same final answer.
- The **same lifecycle events, in the same order**, appear under JSON and RPC.
- What differs is the **observer**: print has no events, JSON has no responses, RPC has both.
- The **failure contract differs** by interface: print exits non-zero, JSON carries the failure in the stream and exits 0, RPC's command succeeds with the failure in the stream.
- The harness spawns the entry the package publishes as `pi` — the **shipped binary**, not a module inside `dist/`.
- A fresh **1.0.4 trace** is regenerated from the chapter's own exporter and compared with the published **1.0.2 recording**.

## What this notebook does **not** establish

- **Recorded inspection and fresh execution are different.** The published recording is 1.0.2 and stays labelled 1.0.2.
- **A browser replay is an inspection interface**, not proof that code ran in a browser or that a real model was called.
- **The model is scripted** — this measures the code around the model, not model behaviour.
- **The recording may be absent** from this checkout; if so the notebook reports it NOT INSPECTED instead of inventing it.


## Setup

This notebook runs the book's own examples. **Pi is Node/TypeScript**; Python
only presents, orchestrates and checks. There is no Python re-implementation
of Pi's loop, context loader, session projection, permissions or protocol
anywhere in this set — where a mechanism is demonstrated, the code that runs is
the code in `examples/`.

* **Execution mode:** **Run** + **Inspect** — a fresh 1.0.4 run through the shipped binary, a fresh 1.0.4 trace from the chapter's exporter, and (when present) the published 1.0.2 recording.
* **Evidence scope:** `shipped_binary`. The model is scripted; no real model is called.
* **Credentials:** none. The model is a scripted stand-in (`faux`) supplied by
  the book's harness, so nothing here reaches the network, spends money, or
  says anything about how a real model behaves.


In [1]:
# --- Locate the shared support module and the pinned packages -----------------
# Two documented markers, both overridable:
#   PIN_REPO       -> the checkout of the pi repository
#   PIN_NOTEBOOKS  -> this notebook directory
# Because both searches walk upwards from the working directory, this cell
# works whether you launched Jupyter from notebooks/pi/ or from the repo root.
import sys
from pathlib import Path


def _find_support() -> Path:
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "pinb.py").is_file():
            return c
        nested = c / "notebooks" / "pi"
        if (nested / "pinb.py").is_file():
            return nested
    raise SystemExit(
        "Could not find notebooks/pi/pinb.py from " + str(Path.cwd()) + ".\n"
        "Open this notebook from notebooks/pi/, or set PIN_NOTEBOOKS."
    )


sys.path.insert(0, str(_find_support()))
import pinb  # noqa: E402  (a local module, deliberately after the path fix)

pinb.self_check()


{'repo_root': 'resolved',
 'notebook_dir': 'resolved',
 'pin_matches_manuscript': True,
 'node': 'v25.8.2',
 'python': '3.13.15',
 'claim_ledger': True,
 'credentials_required': False}

In [2]:
pinb.show_environment("This notebook's runtime")

print()
print("Repository marker: an ancestor directory holding both")
print("  content/books/pi/                          (the manuscript)")
print("  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)")
print("Every path below is shown relative to that repository; no absolute")
print("path is printed, so the saved output is portable.")
print()
print(pinb.md_table(
    ["resource", "repository-relative path"],
    [["manuscript", pinb.rel(pinb.chapter_file(31))],
     ["examples tree", "examples/"],
     ["claim ledger", "examples/evidence.json"],
     ["pinned docs", "examples/node_modules/@earendil-works/pi-coding-agent/docs/"],
     ["pinned declarations", "examples/node_modules/@earendil-works/*/dist/"],
     ["support module", "notebooks/pi/pinb.py"]],
))


== This notebook's runtime ==
component  value
-------------------------------  ----------------------------------------------------
manuscript pin (AGENTS.md)  1.0.4
@earendil-works/pi-ai  1.0.4
@earendil-works/pi-agent-core  1.0.4
@earendil-works/pi-coding-agent  1.0.4
node  v25.8.2
python  3.13.15
platform  win32
examples tree  installed
model used  faux (scripted) - no network, no credential, no cost

Repository marker: an ancestor directory holding both
  content/books/pi/                          (the manuscript)
  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)
Every path below is shown relative to that repository; no absolute
path is printed, so the saved output is portable.

| resource | repository-relative path |
|---|---|
| manuscript | content/books/pi/31-chapter.md |
| examples tree | examples/ |
| claim ledger | examples/evidence.json |
| pinned docs | examples/node_modules/@earendil-works/pi-coding-agent/docs/ |
| pinned declarations | examples/node_mod

## Baseline: one agent, three drivers

The same scripted run — a `bash` call then an answer — through print, JSON and
RPC on the shipped binary.

In [3]:
THREE = pinb.driver_source("ch31-three.ts")
three = pinb.driver(THREE, label="ch31-three", timeout=420)
s = three["success"]
print(pinb.md_table(
    ["driver", "final answer", "exit", "tool wrote file?", "has events?", "has responses?"],
    [["print", s["print"]["text"], s["print"]["exit"], "yes" if s["print"]["toolWroteFile"] else "no", "no", "no"],
     ["JSON", s["json"]["text"], s["json"]["exit"], "yes" if s["json"]["toolWroteFile"] else "no", "yes" if s["json"]["hasEvents"] else "no", "yes" if s["json"]["hasResponses"] else "no"],
     ["RPC", s["rpc"]["text"], s["rpc"]["exit"], "yes" if s["rpc"]["toolWroteFile"] else "no", "yes" if s["rpc"]["hasEvents"] else "no", "yes" if s["rpc"]["hasResponses"] else "no"]],
))

| driver | final answer | exit | tool wrote file? | has events? | has responses? |
|---|---|---|---|---|---|
| print | the answer is forty-two | 0 | yes | no | no |
| JSON | the answer is forty-two | 0 | yes | yes | no |
| RPC | the answer is forty-two | 0 | yes | yes | yes |


In [4]:
pinb.show_checks([
    pinb.check("all three drivers give the same final answer",
               s["print"]["text"] == s["json"]["text"] == s["rpc"]["text"] == "the answer is forty-two",
               "same answer"),
    pinb.check("the tool ran under every driver (ran.flag exists)",
               s["print"]["toolWroteFile"] and s["json"]["toolWroteFile"] and s["rpc"]["toolWroteFile"],
               "tool ran three times"),
    pinb.check("print has no events and no responses",
               not s["print"]["hasEvents"] and not s["print"]["hasResponses"], "one string"),
    pinb.check("JSON has events and no responses",
               s["json"]["hasEvents"] and not s["json"]["hasResponses"], "events only"),
    pinb.check("RPC has both events and responses",
               s["rpc"]["hasEvents"] and s["rpc"]["hasResponses"], "both observers"),
])

== Assertions ==
  assertion  observed
----  -------------------------------------------------  --------------------
PASS  all three drivers give the same final answer  same answer
PASS  the tool ran under every driver (ran.flag exists)  tool ran three times
PASS  print has no events and no responses  one string
PASS  JSON has events and no responses  events only
PASS  RPC has both events and responses  both observers

5/5 assertions held.


## The same lifecycle events, in the same order

In [5]:
print("JSON lifecycle:", " -> ".join(s["json"]["lifecycle"]))
print("RPC lifecycle: ", " -> ".join(s["rpc"]["lifecycle"]))
print()
pinb.show_checks([
    pinb.check("JSON and RPC see the same lifecycle events in the same order",
               s["json"]["lifecycle"] == s["rpc"]["lifecycle"], "identical"),
    pinb.check("the tool executed in both",
               "tool_execution_end" in s["json"]["lifecycle"], "tool_execution_end present"),
    pinb.check("RPC sends a response before the first lifecycle event (accepted, not finished)",
               s["rpc"]["responseBeforeFirstEvent"], "response first"),
])

JSON lifecycle: agent_start -> turn_start -> tool_execution_start -> tool_execution_end -> turn_end -> turn_start -> turn_end -> agent_end -> agent_settled
RPC lifecycle:  agent_start -> turn_start -> tool_execution_start -> tool_execution_end -> turn_end -> turn_start -> turn_end -> agent_end -> agent_settled

== Assertions ==
  assertion  observed
----  --------------------------------------------------------------------------  --------------------------
PASS  JSON and RPC see the same lifecycle events in the same order  identical
PASS  the tool executed in both  tool_execution_end present
PASS  RPC sends a response before the first lifecycle event (accepted, not fini…  response first

3/3 assertions held.


## The failure contract differs by interface

The same provider failure, three ways. This is the one thing the interface
changes.

In [6]:
f = three["failure"]
print(pinb.md_table(
    ["driver", "exit", "where the failure is"],
    [["print", f["print"]["exit"], "on stderr, no stdout" if f["print"]["messageOnStderr"] else "?"],
     ["JSON", f["json"]["exit"], f"in the stream (stopReason={f['json']['streamStopReason']})"],
     ["RPC", "-", f"in the stream (stopReason={f['rpc']['streamStopReason']}); command success={f['rpc']['promptCommandSucceeded']}"]],
))

| driver | exit | where the failure is |
|---|---|---|
| print | 1 | on stderr, no stdout |
| JSON | 0 | in the stream (stopReason=error) |
| RPC | - | in the stream (stopReason=error); command success=True |


In [7]:
pinb.show_checks([
    pinb.check("print mode exits non-zero on a failed response",
               f["print"]["exit"] != 0, f"exit {f['print']['exit']}"),
    pinb.check("print mode writes nothing to stdout",
               f["print"]["stdoutEmpty"], "stdout empty"),
    pinb.check("JSON mode exits 0 and carries the failure in the stream",
               f["json"]["exit"] == 0 and f["json"]["streamStopReason"] == "error",
               "exit 0, stopReason error"),
    pinb.check("RPC command succeeds with the failure in the stream",
               f["rpc"]["promptCommandSucceeded"] and f["rpc"]["streamStopReason"] == "error",
               "response success=true, stopReason error"),
])

== Assertions ==
  assertion  observed
----  -------------------------------------------------------  ---------------------------------------
PASS  print mode exits non-zero on a failed response  exit 1
PASS  print mode writes nothing to stdout  stdout empty
PASS  JSON mode exits 0 and carries the failure in the stream  exit 0, stopReason error
PASS  RPC command succeeds with the failure in the stream  response success=true, stopReason error

4/4 assertions held.


## A fresh 1.0.4 trace from the chapter's exporter

`experiences/ch31.ts` is a real exporter: it runs the scripted scenario against
the shipped binary and writes a `book-evidence-trace/1` envelope. Nothing here
writes a trace by hand.

In [8]:
TRACE = pinb.driver_source("ch31-trace.ts")
trace = pinb.driver(TRACE, label="ch31-trace", timeout=420)
print(pinb.md_table(
    ["field", "value"],
    [["schema", trace["schema"]],
     ["result", trace["result"]],
     ["evidenceScope", trace["evidenceScope"]],
     ["claimClass", trace["claimClass"]],
     ["recordedAt", trace["recordedAt"]],
     ["runs", ", ".join(r["label"] for r in trace["runs"])],
     ["environment pi-ai", trace["environment"].get("pi-ai", "?")]],
))

| field | value |
|---|---|
| schema | book-evidence-trace/1 |
| result | PASS |
| evidenceScope | shipped_binary |
| claimClass | OBSERVED |
| recordedAt | 2026-10-07T21:28:55.338Z |
| runs | Print mode, success, JSON mode, success, RPC, success, Print mode, failed response, JSON mode, failed response, RPC, failed response |
| environment pi-ai | 1.0.4 |


In [9]:
pinb.show_checks([
    pinb.check("the exporter produces a schema-valid trace",
               trace["schema"] == "book-evidence-trace/1", "schema ok"),
    pinb.check("the fresh trace records the pinned pi-ai version",
               trace["environment"].get("pi-ai") == pinb.PIN, f"pi-ai {trace['environment'].get('pi-ai')}"),
    pinb.check("the fresh trace passes its own observations",
               trace["result"] == "PASS", "PASS"),
    pinb.check("the trace is labelled OBSERVED",
               trace["claimClass"] == "OBSERVED", "OBSERVED"),
])

== Assertions ==
  assertion  observed
----  ------------------------------------------------  -----------
PASS  the exporter produces a schema-valid trace  schema ok
PASS  the fresh trace records the pinned pi-ai version  pi-ai 1.0.4
PASS  the fresh trace passes its own observations  PASS
PASS  the trace is labelled OBSERVED  OBSERVED

4/4 assertions held.


## The published recording (historical, Pi 1.0.2)

The browser experience at `/tools/ai/pi/31-chapter/` replays a recording captured
on **Pi 1.0.2**. It is evidence about that release, not about this pin. If its
directory is present it is inspected and labelled; if not, the notebook says so
rather than inventing it.

Set `PIN_EXPERIENCES` to the site repo's `content/tools/ai/pi` to inspect it.

In [10]:
import json
exp = pinb.experience_dir(31)
if exp is None:
    pinb.unrun("the published 1.0.2 recording", "not present in this checkout; set PIN_EXPERIENCES to the site repo's content/tools/ai/pi")
else:
    rec = pinb.read_trace(exp / "recorded-trace.json")
    meta = json.loads((exp / "experiment.json").read_text(encoding="utf-8"))
    print("Published experience:", meta["id"])
    print("Pinned version:", meta["pinnedVersion"])
    print("Recorded at:", rec.get("recordedAt"))
    print("Environment at capture:", rec.get("environment"))
    print()
    print("This is a RECORDED inspection of 1.0.2 evidence, not a fresh run.")
    pinb.show_checks([
        pinb.check("the recording is clearly labelled 1.0.2",
                   meta["pinnedVersion"] == "1.0.2", "labelled 1.0.2"),
        pinb.check("the recording is NOT relabelled to the current pin",
                   rec.get("environment", {}).get("pi-ai") != pinb.PIN or meta["pinnedVersion"] == pinb.PIN,
                   "not silently re-grounded"),
    ])

Published experience: pi/chapter-31/one-agent-three-drivers
Pinned version: 1.0.2
Recorded at: 2026-10-05T07:59:27.354Z
Environment at capture: {'pi-ai': '1.0.2', 'pi-agent-core': '1.0.2', 'pi-coding-agent': '1.0.2', 'node': 'v25.8.2', 'platform': 'win32'}

This is a RECORDED inspection of 1.0.2 evidence, not a fresh run.
== Assertions ==
  assertion  observed
----  --------------------------------------------------  ------------------------
PASS  the recording is clearly labelled 1.0.2  labelled 1.0.2
PASS  the recording is NOT relabelled to the current pin  not silently re-grounded

2/2 assertions held.


## The chapter's own tests

All six shipped-binary tests. They exercise the public bundled executable, which
is the strongest evidence this chapter has.

## The chapter's canonical evidence (shipped binary)

These are the chapter's own tests, executed unmodified through Node's test
runner. Every ledger row the chapter quotes comes from a named case in this
file, so running it here is running the chapter's evidence, not a copy of it.

In [11]:
PATTERNS = 'ch31-interface/binary.test.ts'
tap = pinb.run_tests(PATTERNS, timeout=420)
pinb.show_tap(tap, files='ch31-interface/binary.test.ts', show="interface")

$ node --test ch31-interface/binary.test.ts
  PASS  5 passed, 0 failed, 0 skipped  in 3673 ms

  tests matching 'interface':
    ok   the agent did the same thing under each interface: the same events, in the same order
    ok   the failure contract differs by interface, and nothing else about the run does


## Your turn: predict, then run

**Predict first.** Print mode exits non-zero on a failed response. What does it
exit with on a *successful* run — 0, or the model's stop reason?

**Then change one input.** The script runs a `bash` tool. Remove it and run
`ch31-three.ts` again with only `{ text: ... }`. Does the lifecycle lose
`tool_execution_*` in all three drivers?

**Predict the boundary.** RPC's command succeeds even when the model failed.
What would a client that only checks `success` do wrong here?

In [12]:
print("Predictions:")
print("  1. A successful print run exits 0.")
print("  2. Without the tool, tool_execution_* disappears from JSON and RPC alike.")
print("  3. A client checking only success would report a failed run as fine.")
print()
print("Observed above:")
print(f"  success: print exit={s['print']['exit']}, JSON exit={s['json']['exit']}, RPC exit={s['rpc']['exit']}")
print(f"  answers: {s['print']['text']} | {s['json']['text']} | {s['rpc']['text']}")
print(f"  failure: print exit={f['print']['exit']}, JSON exit={f['json']['exit']}, RPC success={f['rpc']['promptCommandSucceeded']}")
print()
assert s["print"]["text"] == s["json"]["text"] == s["rpc"]["text"]
assert s["json"]["lifecycle"] == s["rpc"]["lifecycle"]
assert f["print"]["exit"] != 0 and f["json"]["exit"] == 0
assert f["rpc"]["promptCommandSucceeded"] and f["rpc"]["streamStopReason"] == "error"
print("held: same agent under three drivers; the interface changes only the observer and the failure contract")

Predictions:
  1. A successful print run exits 0.
  2. Without the tool, tool_execution_* disappears from JSON and RPC alike.
  3. A client checking only success would report a failed run as fine.

Observed above:
  success: print exit=0, JSON exit=0, RPC exit=0
  answers: the answer is forty-two | the answer is forty-two | the answer is forty-two
  failure: print exit=1, JSON exit=0, RPC success=True

held: same agent under three drivers; the interface changes only the observer and the failure contract


## Interpretation

The interface is an **observer**, and the chapter shows what each one can see:

| Driver | Output | Events | Responses | Failure contract |
|---|---|---|---|---|
| **print** | Final text | No | No | Non-zero exit, stderr |
| **JSON** | Event stream | Yes | No | Exit 0, failure in the stream |
| **RPC** | Line-framed JSONL | Yes | Yes | Command succeeds, failure in the stream |

The practical rules:
1. **The interface does not change the agent.** Same answer, same lifecycle events.
2. **Choose by what you need to observe** — a final answer (print), events (JSON), or a control channel (RPC).
3. **Check the failure contract.** A script that only checks an exit status will miss a JSON-mode failure; a client that only checks `success` will miss an RPC-mode one.
4. **Recorded is not fresh.** The published experience is a 1.0.2 recording and stays labelled that way.
5. **A browser replay is an inspection interface** — it is not proof that code ran in a browser or that a real model was called.

## Sources, execution mode and limitations

**Execution mode:** **Run** + **Inspect** — a fresh 1.0.4 run through the shipped binary, a fresh 1.0.4 trace from the chapter's exporter, and (when present) the published 1.0.2 recording.

**Evidence scope:** `shipped_binary`. The model is scripted; no real model is called.

### What was read or run

- **Ran** `drivers/ch31-three.ts`: the same scripted run and the same failure through print, JSON and RPC on the shipped binary.
- **Ran** `drivers/ch31-trace.ts`: `experiences/ch31.ts` regenerated a fresh 1.0.4 `book-evidence-trace/1` envelope.
- **Inspected** the published 1.0.2 recording at `/tools/ai/pi/31-chapter/` when `PIN_EXPERIENCES` resolves it (recorded, not fresh).
- **Ran** `ch31-interface/binary.test.ts` (6 tests).
- **Read** `examples/evidence.json`, chapter 31 rows (6 claims).

### Limitations

- **The published recording is 1.0.2** and is not relabelled.
- **A browser replay is an inspection interface**, not proof of execution.
- **The model is scripted** — no real-model behaviour is measured.

### Optional work that was NOT run

- The published browser experience is inspected only when its directory is present; otherwise it is reported NOT INSPECTED.

Every technical statement in this notebook is about Pi **1.0.4**. A later
release can change any of it; the book's drift protocol in `AGENTS.md` is the
procedure for finding out whether it did.
